# 03.1 — `Dataset` vs `fileDatastore`

Module 03 walks the data pipeline — the path from `.mat` files on disk to batched tensors in the training loop. It starts here, with the two objects that own "give me trial number k": MATLAB's `fileDatastore` and PyTorch's `Dataset`.

This notebook covers:

* The `fileDatastore` model you already know, mapped piece-by-piece onto PyTorch.
* The `Dataset` protocol — two dunder methods and nothing else.
* Lazy loading: why neither system holds the corpus in RAM.
* This project's two datasets: `SyntheticTrialDataset` and `MatFileTrialDataset`.

**Prerequisites:** [01.4 classes](../01_python_for_matlab_users/01.4_classes_and_oop.ipynb) (the dunder protocol), [02.2 axis conventions](../02_numpy_and_pytorch_basics/02.2_array_axis_conventions.ipynb).

## Section 1 — What MATLAB does

The MATLAB pipeline (`cgg_procAutoEncoder.m`) wires the corpus like this:

```matlab
Data_Fun   = @(x) cgg_loadDataArray(x, DataWidth, ..., 'Normalization', Normalization, ...);
Target_Fun = @(x) cgg_loadTargetArray(x, 'Dimension', Dimension);

Data_ds   = fileDatastore(DataAggregateDir,   "ReadFcn", Data_Fun);
Target_ds = fileDatastore(TargetAggregateDir, "ReadFcn", Target_Fun);
DataStore = combine(Data_ds, Target_ds, DataNumber_ds);
```

Three moving parts:

1. **A directory of files** — one `.mat` per trial.
2. **A read function** — how to turn one file into one training example (windowing, target selection… the whole recipe is curried into the function handle).
3. **The datastore object** — hands out examples one `read()` at a time, lazily.

PyTorch's `Dataset` is the same three parts wearing a class instead of a function handle.

## Section 2 — The Python concepts you need

### 2.1 — The protocol: `__len__` + `__getitem__`

A map-style `Dataset` is any class with two methods: *how many examples* and *give me example k*. That's the entire contract:

In [ ]:
import torch
from torch.utils.data import Dataset

class SquaresDataset(Dataset):
    """A toy dataset: example k is (k, k²) — no files, no I/O, pure protocol."""

    def __init__(self, n: int):
        self.n = n

    def __len__(self):
        return self.n

    def __getitem__(self, idx):
        if not 0 <= idx < self.n:
            raise IndexError(idx)
        return torch.tensor([float(idx)]), torch.tensor([float(idx**2)])

ds = SquaresDataset(100)
print(len(ds))          # __len__
print(ds[7])            # __getitem__ — MATLAB analog: read the 8th file (0-indexing!)

The mapping to `fileDatastore`:

| fileDatastore | Dataset | Notes |
|---|---|---|
| the file list it scans | whatever `__init__` discovers | discovery runs ONCE, up front |
| `ReadFcn` | the body of `__getitem__` | runs per access — this is where lazy work lives |
| `read(ds)` / `hasdata(ds)` | `ds[idx]` / `len(ds)` | random access instead of sequential-with-cursor |
| `reset(ds)` | (nothing) | no cursor, nothing to reset |
| `combine(ds1, ds2)` | return a tuple from `__getitem__` | one dataset returns (x, target, metadata) together |

Two design consequences worth absorbing:

* **Random access replaces the cursor.** A datastore is a stream you walk forward; a `Dataset` is indexable. Shuffling becomes trivial (permute the indices — the DataLoader's job, [03.2](03.2_dataloader_and_collation.ipynb)) instead of `shuffle(ds)` on the store.
* **`combine` disappears.** Where MATLAB zips three datastores (data + target + trial number), the Python `__getitem__` just returns all three pieces at once — that's the `(x, targets, metadata)` triple every dataset in this project yields.

### 2.2 — Lazy vs eager, and where work belongs

The rule both systems share: **discovery in the constructor, I/O in the read path.**

In [ ]:
import time

class EagerDataset(Dataset):
    """Anti-pattern: loads EVERYTHING in __init__."""
    def __init__(self, n):
        time.sleep(0.5)                       # imagine: reading n .mat files
        self.data = [torch.randn(100) for _ in range(n)]
    def __len__(self): return len(self.data)
    def __getitem__(self, i): return self.data[i]

class LazyDataset(Dataset):
    """The pattern: __init__ only discovers; __getitem__ loads."""
    def __init__(self, n):
        self.n = n                            # imagine: globbing filenames only
    def __len__(self): return self.n
    def __getitem__(self, i):
        return torch.randn(100)               # imagine: loading ONE .mat here

t0 = time.perf_counter(); EagerDataset(64); t_eager = time.perf_counter() - t0
t0 = time.perf_counter(); LazyDataset(64);  t_lazy  = time.perf_counter() - t0
print(f"eager construction: {t_eager:.3f}s   lazy construction: {t_lazy:.6f}s")

For this project the lazy rule is load-bearing: a session's trials are hundreds of multi-megabyte `.mat` files. `MatFileTrialDataset` globs filenames and reads targets (small) up front, but each trial's data file is opened only inside `__getitem__` — RAM stays flat no matter the corpus size, exactly like `fileDatastore`. (An opt-in `preload=True` trades memory for speed when the corpus fits.)

### 2.3 — This project's datasets, live

In [ ]:
from neural_data_decoding.data import SyntheticTrialDataset

ds = SyntheticTrialDataset(
    num_sessions=3, trials_per_session=8,
    num_samples=10,          # W — windows per trial
    num_features=4,          # C — channels
    num_classes_per_dim=[3, 2],
    seed=0,
)
print("len:", len(ds))
x, targets, meta = ds[0]
print("x:", tuple(x.shape), "| targets:", targets.tolist(), "| meta:", meta)

In [ ]:
# The real-data twin — same protocol, same triple, backed by .mat files.
from pathlib import Path
from neural_data_decoding.data import MatFileTrialDataset

FIXTURE_DIR = Path("../../results/Decision")
if (FIXTURE_DIR / "Decision_Data_0000011.mat").is_file():
    real = MatFileTrialDataset(
        data_dir=FIXTURE_DIR, target_dir=FIXTURE_DIR,
        data_width=100, window_stride=50,
    )
    x, targets, meta = real[0]
    print("len:", len(real), "| x:", tuple(x.shape), "(W, T, A, C)")
    print("targets:", targets.tolist(), "| meta:", meta)
else:
    print("fixture not present — the synthetic dataset above shows the identical protocol")

Same `(x, targets, metadata)` contract from both — which is the point: **the training loop cannot tell them apart.** Swapping synthetic for real data (what the CLI does when `cfg.data_dir` is set) changes one constructor call and nothing downstream.

## Section 3 — The `neural_data_decoding` implementation

The `fileDatastore`-analog anatomy of `MatFileTrialDataset` — discovery up front, pairing by filename suffix (the analog of MATLAB's two-directory `combine`):

In [ ]:
src = Path("../../src/neural_data_decoding/data/mat_dataset.py").read_text().splitlines()
i = next(j for j, line in enumerate(src) if line.startswith("def _discover_trials"))
for k in range(i, min(i + 18, len(src))):
    print(f"{k + 1:4} | {src[k]}")

Things to spot:

* Discovery is a **module-level function**, not a method — testable in isolation (and it is: `tests/unit/test_mat_dataset.py` covers the unpaired-file error path).
* Pairing validates BOTH directions (data without target, target without data) and fails loudly with the offending IDs — the MATLAB `combine` would silently zip mismatched orders.
* The windowing math itself lives in `_compute_window_starts`, a direct port of `cgg_loadDataArray.m` lines 130–184 — the docstrings cite the exact MATLAB lines they mirror.

## Section 4 — Hands-on exercises

### Exercise 4.1 — write a Dataset

Implement `EveryOtherDataset`, wrapping any dataset but exposing only its even-indexed examples. (This is how subset-views work — compare `torch.utils.data.Subset`.)

In [ ]:
# Your attempt here


In [ ]:
# Reveal:
class EveryOtherDataset(Dataset):
    def __init__(self, base):
        self.base = base
    def __len__(self):
        return (len(self.base) + 1) // 2
    def __getitem__(self, i):
        return self.base[2 * i]

half = EveryOtherDataset(SquaresDataset(10))
print(len(half), half[3])     # 5 examples; half[3] is base[6] → (6, 36)

### Exercise 4.2 — read the MATLAB, find the Python

`cgg_procAutoEncoder.m` builds TWO data datastores over the same directory: `Data_ds` (clean reads) and `Data_Augmented_ds` (reads with noise injection), switching the training store to the augmented read function. Which constructor argument of `MatFileTrialDataset` plays the role of that switch? (Check the class docstring — answer in [03.6](03.6_augmentation_per_call_contract.ipynb).)

## Section 5 — Common errors

### `TypeError: object of type 'MyDataset' has no len()`

You forgot `__len__`. Map-style datasets need both methods — the DataLoader asks for the length up front to plan an epoch.

### `KeyError`/`TypeError` inside `__getitem__` when the DataLoader runs but `ds[0]` works

The DataLoader may pass indices as different int types, and with workers, exceptions get re-raised with the worker's traceback attached. Test `ds[len(ds) - 1]` and `ds[0]` directly first — direct indexing reproduces most bugs without the worker noise.

### Training is mysteriously slow, disk thrashing

Heavy work in the wrong place — e.g., re-globbing the directory or re-reading a shared file inside `__getitem__`. Discovery once in `__init__`; per-example I/O only in `__getitem__`.

### Out of RAM constructing the dataset

The eager anti-pattern (§2.2): loading the corpus in `__init__`. Keep construction to filenames + small metadata.

### Everything works, but examples repeat within an epoch

Your `__len__` disagrees with reality (too large → wraparound via modulo-style bugs in custom code, or IndexErrors; too small → silently unused data). Assert `len(ds)` against the file count.

## Section 6 — Further reading

- [torch.utils.data docs](https://pytorch.org/docs/stable/data.html) — the full Dataset/DataLoader reference, including iterable-style datasets (the closer analog to a pure stream, rarely needed here).
- [MATLAB fileDatastore docs](https://www.mathworks.com/help/matlab/ref/matlab.io.datastore.filedatastore.html) — for the side-by-side.
- [`src/neural_data_decoding/data/mat_dataset.py`](../../src/neural_data_decoding/data/mat_dataset.py) — the production read path, docstrings first.

Next up: **[03.2 DataLoader and collation](03.2_dataloader_and_collation.ipynb)** — from single examples to batched tensors.